# 🛰️ Nexus_SIH: Sentinel-2 Multi-Spectral SRGAN End-to-End Training

Welcome to the **Nexus SIH** Super-Resolution pipeline!

This notebook is **100% self-contained**:
- Runs directly on Google Colab's free **NVIDIA T4 GPU**.
- Generates/slices multi-spectral 4-band satellite data (Red, Green, Blue, Near-Infrared).
- Trains the Generator and Discriminator with Spectral Angle Mapper (SAM) loss for NDVI preservation.
- Computes PSNR, SSIM, SAM, and renders an **AI Uncertainty Map**.
- Saves and downloads `generator_best.pth` and `srgan_sentinel2_x4.pth` directly to your local computer.

### ⚡ Step 1: Verify GPU Acceleration
Make sure your runtime is using GPU: **Runtime** -> **Change runtime type** -> **T4 GPU** -> **Save**.

In [ ]:
!nvidia-smi

### 📦 Step 2: Install Supporting Dependencies

In [ ]:
!pip install -q pyyaml tifffile scikit-image rasterio matplotlib

### ✂️ Step 3: Dataset Preparation & Slicing
Creates High-Resolution (256x256) and matching Low-Resolution (64x64) 4-band patches.
If you have your own Sentinel-2 `.tif` files, you can place them in `data/raw/sentinel2/`.
Otherwise, the cell generates realistic multi-spectral 4-band terrain simulation tiles (RGB + NIR vegetation fields) so you can train immediately!

In [ ]:
import os
import glob
import numpy as np
from pathlib import Path
from PIL import Image
import torch
from torch.utils.data import Dataset, DataLoader

# Create directories
train_hr_dir = Path('data/processed/train/hr')
train_lr_dir = Path('data/processed/train/lr')
val_hr_dir = Path('data/processed/val/hr')
val_lr_dir = Path('data/processed/val/lr')
checkpoints_dir = Path('checkpoints')
outputs_dir = Path('data/outputs')

for d in [train_hr_dir, train_lr_dir, val_hr_dir, val_lr_dir, checkpoints_dir, outputs_dir]:
    d.mkdir(parents=True, exist_ok=True)

def generate_synthetic_sentinel_tile(h=1024, w=1024, num_bands=4):
    x = np.linspace(0, 10, w)
    y = np.linspace(0, 10, h)
    xx, yy = np.meshgrid(x, y)
    terrain = np.sin(xx * 0.5) * np.cos(yy * 0.5) + 0.5 * np.sin(xx * 1.5 + yy * 1.2)
    terrain = (terrain - terrain.min()) / (terrain.max() - terrain.min() + 1e-6)
    fields = ((np.sin(xx * 3.0) > 0.1) * (np.cos(yy * 3.0) > 0.1)).astype(np.float32)

    tile = np.zeros((num_bands, h, w), dtype=np.float32)
    tile[0] = np.clip(0.3 * terrain + 0.2 * fields + np.random.normal(0, 0.02, (h, w)), 0.05, 0.95) # Red (B04)
    tile[1] = np.clip(0.35 * terrain + 0.3 * fields + np.random.normal(0, 0.02, (h, w)), 0.05, 0.95) # Green (B03)
    tile[2] = np.clip(0.25 * terrain + 0.1 * fields + np.random.normal(0, 0.02, (h, w)), 0.05, 0.95) # Blue (B02)
    tile[3] = np.clip(0.4 * terrain + 0.5 * fields + np.random.normal(0, 0.02, (h, w)), 0.05, 0.95) # NIR (B08)
    return tile

def downsample_patch(patch_c_h_w, scale=4):
    c, h, w = patch_c_h_w.shape
    lr_h, lr_w = h // scale, w // scale
    lr = np.zeros((c, lr_h, lr_w), dtype=np.float32)
    for i in range(c):
        band_img = Image.fromarray((patch_c_h_w[i] * 255).astype(np.uint8))
        band_lr = band_img.resize((lr_w, lr_h), Image.Resampling.BICUBIC)
        lr[i] = np.array(band_lr, dtype=np.float32) / 255.0
    return lr

print('✂️ Slicing Sentinel-2 multi-spectral dataset...')
hr_size, scale = 256, 4
total_patches = 0
for i in range(4):
    tile = generate_synthetic_sentinel_tile(1024, 1024, num_bands=4)
    _, h, w = tile.shape
    for y in range(0, h - hr_size + 1, hr_size):
        for x in range(0, w - hr_size + 1, hr_size):
            hr_patch = tile[:, y:y + hr_size, x:x + hr_size]
            lr_patch = downsample_patch(hr_patch, scale=scale)
            is_val = (total_patches % 5 == 0)
            hr_dst = val_hr_dir if is_val else train_hr_dir
            lr_dst = val_lr_dir if is_val else train_lr_dir
            filename = f'tile_{i}_p{total_patches:04d}.npy'
            np.save(hr_dst / filename, hr_patch)
            np.save(lr_dst / filename, lr_patch)
            total_patches += 1

print(f'✅ Dataset ready! Training patches: {len(list(train_hr_dir.glob("*.npy")))} | Validation patches: {len(list(val_hr_dir.glob("*.npy")))}')

# PyTorch Dataset
class SatelliteSRDataset(Dataset):
    def __init__(self, lr_dir, hr_dir, in_channels=4):
        self.lr_files = sorted(list(Path(lr_dir).glob('*.npy')))
        self.hr_dir = Path(hr_dir)
        self.in_channels = in_channels

    def __len__(self):
        return len(self.lr_files)

    def __getitem__(self, idx):
        lr_file = self.lr_files[idx]
        hr_file = self.hr_dir / lr_file.name
        lr_arr = np.load(lr_file).astype(np.float32)[:self.in_channels]
        hr_arr = np.load(hr_file).astype(np.float32)[:self.in_channels]
        return torch.from_numpy(lr_arr), torch.from_numpy(hr_arr)

train_loader = DataLoader(SatelliteSRDataset(train_lr_dir, train_hr_dir), batch_size=8, shuffle=True)
val_loader = DataLoader(SatelliteSRDataset(val_lr_dir, val_hr_dir), batch_size=1, shuffle=False)

### 🧠 Step 4: Define Sentinel-2 SRGAN Architecture & Multi-Spectral Losses

In [ ]:
import torch.nn as nn

# Generator Residual Block
class ResidualBlock(nn.Module):
    def __init__(self, channels=64):
        super().__init__()
        self.conv1 = nn.Conv2d(channels, channels, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(channels)
        self.prelu = nn.PReLU()
        self.conv2 = nn.Conv2d(channels, channels, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(channels)

    def forward(self, x):
        return x + self.bn2(self.conv2(self.prelu(self.bn1(self.conv1(x)))))

# SRGAN Generator (4x Upscale via PixelShuffle)
class SRGANGenerator(nn.Module):
    def __init__(self, in_c=4, out_c=4, n_res_blocks=16, scale=4):
        super().__init__()
        self.head = nn.Sequential(
            nn.Conv2d(in_c, 64, kernel_size=9, padding=4),
            nn.PReLU()
        )
        self.body = nn.Sequential(*[ResidualBlock(64) for _ in range(n_res_blocks)])
        self.trunk = nn.Sequential(
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64)
        )
        self.upsample = nn.Sequential(
            nn.Conv2d(64, 256, kernel_size=3, padding=1),
            nn.PixelShuffle(2),
            nn.PReLU(),
            nn.Conv2d(64, 256, kernel_size=3, padding=1),
            nn.PixelShuffle(2),
            nn.PReLU()
        )
        self.tail = nn.Conv2d(64, out_c, kernel_size=9, padding=4)

    def forward(self, x):
        h = self.head(x)
        b = self.body(h)
        t = self.trunk(b) + h
        u = self.upsample(t)
        return torch.sigmoid(self.tail(u))

# Discriminator Block
class DiscriminatorBlock(nn.Module):
    def __init__(self, in_c, out_c, stride=1, normalize=True):
        super().__init__()
        layers = [nn.Conv2d(in_c, out_c, kernel_size=3, stride=stride, padding=1)]
        if normalize:
            layers.append(nn.BatchNorm2d(out_c))
        layers.append(nn.LeakyReLU(0.2, inplace=True))
        self.block = nn.Sequential(*layers)

    def forward(self, x):
        return self.block(x)

# SRGAN Discriminator
class SRGANDiscriminator(nn.Module):
    def __init__(self, in_channels=4):
        super().__init__()
        self.net = nn.Sequential(
            DiscriminatorBlock(in_channels, 64, stride=1, normalize=False),
            DiscriminatorBlock(64, 64, stride=2),
            DiscriminatorBlock(64, 128, stride=1),
            DiscriminatorBlock(128, 128, stride=2),
            DiscriminatorBlock(128, 256, stride=1),
            DiscriminatorBlock(256, 256, stride=2),
            DiscriminatorBlock(256, 512, stride=1),
            DiscriminatorBlock(512, 512, stride=2),
            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten(),
            nn.Linear(512, 1024),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Linear(1024, 1)
        )

    def forward(self, x):
        return self.net(x)

# Spectral Angle Mapper (SAM) Loss
class SpectralAngleMapperLoss(nn.Module):
    def __init__(self, eps=1e-7):
        super().__init__()
        self.eps = eps

    def forward(self, sr, hr):
        dot = torch.sum(sr * hr, dim=1)
        norm_sr = torch.norm(sr, p=2, dim=1)
        norm_hr = torch.norm(hr, p=2, dim=1)
        cos_theta = torch.clamp(dot / (norm_sr * norm_hr + self.eps), -1.0 + self.eps, 1.0 - self.eps)
        return torch.mean(torch.acos(cos_theta))

# Gradient / Edge Loss
class GradientEdgeLoss(nn.Module):
    def forward(self, sr, hr):
        dy_sr = torch.abs(sr[:, :, 1:, :] - sr[:, :, :-1, :])
        dx_sr = torch.abs(sr[:, :, :, 1:] - sr[:, :, :, :-1])
        dy_hr = torch.abs(hr[:, :, 1:, :] - hr[:, :, :-1, :])
        dx_hr = torch.abs(hr[:, :, :, 1:] - hr[:, :, :, :-1])
        return torch.mean(torch.abs(dy_sr - dy_hr)) + torch.mean(torch.abs(dx_sr - dx_hr))

print('✅ SRGAN Neural Architectures & Spectral Loss functions initialized!')

### 🚀 Step 5: Start Model Training & Fine-Tuning
Trains on the GPU for 25 epochs, tracking PSNR and saving `checkpoints/generator_best.pth`.

In [ ]:
import time

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'⚡ Training on: {device} ({torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"})')

generator = SRGANGenerator(in_c=4, out_c=4, scale=4).to(device)
discriminator = SRGANDiscriminator(in_channels=4).to(device)

criterion_content = nn.L1Loss().to(device)
criterion_adv = nn.BCEWithLogitsLoss().to(device)
criterion_sam = SpectralAngleMapperLoss().to(device)
criterion_edge = GradientEdgeLoss().to(device)

optimizer_g = torch.optim.Adam(generator.parameters(), lr=1e-4, betas=(0.9, 0.999))
optimizer_d = torch.optim.Adam(discriminator.parameters(), lr=1e-4, betas=(0.9, 0.999))

epochs = 25
best_psnr = 0.0
start_time = time.time()

print(f'🚀 Starting training loop for {epochs} epochs...')
for epoch in range(1, epochs + 1):
    generator.train()
    discriminator.train()
    running_g_loss = 0.0
    running_d_loss = 0.0
    running_psnr = 0.0
    running_sam = 0.0
    count = 0

    for lr_imgs, hr_imgs in train_loader:
        lr_imgs, hr_imgs = lr_imgs.to(device), hr_imgs.to(device)
        bs = lr_imgs.size(0)
        real_labels = torch.ones(bs, 1, device=device)
        fake_labels = torch.zeros(bs, 1, device=device)

        # Train Discriminator
        optimizer_d.zero_grad()
        pred_real = discriminator(hr_imgs)
        d_real_loss = criterion_adv(pred_real, real_labels)
        sr_imgs = generator(lr_imgs)
        pred_fake = discriminator(sr_imgs.detach())
        d_fake_loss = criterion_adv(pred_fake, fake_labels)
        d_loss = (d_real_loss + d_fake_loss) * 0.5
        d_loss.backward()
        optimizer_d.step()

        # Train Generator
        optimizer_g.zero_grad()
        pred_fake_g = discriminator(sr_imgs)
        adv_loss = criterion_adv(pred_fake_g, real_labels)
        content_loss = criterion_content(sr_imgs, hr_imgs)
        sam_loss = criterion_sam(sr_imgs, hr_imgs)
        edge_loss = criterion_edge(sr_imgs, hr_imgs)

        g_loss = content_loss + 0.001 * adv_loss + 0.05 * sam_loss + 0.05 * edge_loss
        g_loss.backward()
        optimizer_g.step()

        with torch.no_grad():
            mse = torch.mean((sr_imgs - hr_imgs) ** 2)
            psnr_val = 20.0 * torch.log10(1.0 / torch.sqrt(mse + 1e-8)).item()

        running_g_loss += g_loss.item()
        running_d_loss += d_loss.item()
        running_sam += sam_loss.item()
        running_psnr += psnr_val
        count += 1

    avg_psnr = running_psnr / count
    avg_sam = running_sam / count
    print(f'Epoch [{epoch:02d}/{epochs:02d}] Loss_G: {running_g_loss/count:.4f} | Loss_D: {running_d_loss/count:.4f} | SAM: {avg_sam:.4f} rad | PSNR: {avg_psnr:.2f} dB')

    if avg_psnr > best_psnr:
        best_psnr = avg_psnr
        torch.save(generator.state_dict(), 'checkpoints/generator_best.pth')
        torch.save(generator.state_dict(), 'checkpoints/srgan_sentinel2_x4.pth')

total_mins = (time.time() - start_time) / 60.0
print(f'\n🎉 Training Complete in {total_mins:.2f} mins! Best PSNR: {best_psnr:.2f} dB')
print('💾 Saved checkpoints to: checkpoints/generator_best.pth')

### 🔍 Step 6: Benchmark Metrics & Generate Uncertainty Map

In [ ]:
from skimage.metrics import structural_similarity as ssim
import matplotlib.pyplot as plt

generator.eval()
psnr_list, ssim_list, sam_list = [], [], []
sample_vis = None

with torch.no_grad():
    for i, (lr_t, hr_t) in enumerate(val_loader):
        lr_t, hr_t = lr_t.to(device), hr_t.to(device)
        sr_t = generator(lr_t)

        lr_np = lr_t.squeeze(0).cpu().numpy()
        hr_np = hr_t.squeeze(0).cpu().numpy()
        sr_np = sr_t.squeeze(0).cpu().numpy()

        mse = np.mean((sr_np - hr_np) ** 2)
        p = 20.0 * np.log10(1.0 / np.sqrt(mse + 1e-8))
        s = ssim(sr_np[0], hr_np[0], data_range=1.0)

        psnr_list.append(p)
        ssim_list.append(s)

        if i == 0:
            uncertainty = np.mean(np.abs(sr_np - hr_np), axis=0)
            sample_vis = (lr_np, sr_np, hr_np, uncertainty)

print('📊 Validation Benchmark Scores:')
print(f'   📈 Mean PSNR: {np.mean(psnr_list):.2f} dB')
print(f'   📈 Mean SSIM: {np.mean(ssim_list):.4f}')

# Plot 4-panel comparison
if sample_vis:
    lr, sr, hr, unc = sample_vis
    fig, axs = plt.subplots(1, 4, figsize=(20, 5))
    axs[0].imshow(np.clip(np.transpose(lr[:3], (1, 2, 0)), 0, 1))
    axs[0].set_title('Input LR (10m)')
    axs[0].axis('off')

    axs[1].imshow(np.clip(np.transpose(sr[:3], (1, 2, 0)), 0, 1))
    axs[1].set_title('SRGAN Super-Res (2.5m)')
    axs[1].axis('off')

    axs[2].imshow(np.clip(np.transpose(hr[:3], (1, 2, 0)), 0, 1))
    axs[2].set_title('Ground Truth HR')
    axs[2].axis('off')

    im = axs[3].imshow(unc, cmap='inferno')
    axs[3].set_title('AI Uncertainty Map')
    axs[3].axis('off')
    plt.colorbar(im, ax=axs[3], fraction=0.046, pad=0.04)

    plt.tight_layout()
    plt.savefig('data/outputs/evaluation_comparison.png', dpi=200)
    plt.show()

### 💾 Step 7: Download Fine-Tuned Model Weights
Downloads `generator_best.pth` directly to your local computer.

In [ ]:
from google.colab import files
import os

best_weights = 'checkpoints/generator_best.pth'
if os.path.exists(best_weights):
    files.download(best_weights)
    print('✅ Downloading generator_best.pth to your machine!')
else:
    print('❌ Checkpoint not found.')